<center><h1>XGBOOST TRAINING PIPELINE</h1><center>

In [ ]:
# ============================================================
#  - Year cap: 2008–2024
#  - Focus genres: Pop, Hip-Hop, Latin, Electronic
#  - Target: median_track_popularity
#  - Features: Audio features (energy, tempo, danceability, etc.)
#  - Time-aware CV: 3y train / 2y test, 1y step, **all folds**
#  - Manual baseline params per genre, then grid per fold; winner by RMSE
#  - Final params per genre: lowest average RMSE across folds
#  - Holdout: train ≤ 2022 → predict 2023–2024
#  - Saves: WF metrics, WF candidates (all tried), holdout results, feature importance, summary
# ============================================================

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import ParameterGrid
from xgboost import XGBRegressor
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# -------------------
# CONFIG (edit these)
# -------------------
YEAR_MIN, YEAR_MAX = 2008, 2024
TRAIN_YEARS, TEST_YEARS = 3, 2
RANDOM_STATE = 42

# Thesis thresholds for reporting (for XGB we'll show them but not hard-pass/fail)
THRESH_RMSE, THRESH_MAE, THRESH_R2 = 8.0, 6.0, 0.70

FOCUS_GENRES = ["pop", "hip-hop", "latin", "electronic"]

# Feature columns in your aggregated CSV (adjust if needed)
FEATURES = [
    "median_energy",
    "median_tempo",
    "median_danceability",
    "median_loudness",
    "median_liveness",
    "median_valence",
    "median_speechiness",
    "median_instrumentalness",
    "median_acousticness",
]
TARGET = "median_track_popularity"

# Manual baseline hyperparameters per genre (you can tweak per genre if you like)
MANUAL_PARAMS = {
    "pop": {
        "max_depth": 4, "learning_rate": 0.08, "n_estimators": 300,
        "subsample": 0.9, "colsample_bytree": 0.9, "reg_lambda": 1.0, "reg_alpha": 0.0
    },
    "hip-hop": {
        "max_depth": 4, "learning_rate": 0.08, "n_estimators": 300,
        "subsample": 0.9, "colsample_bytree": 0.9, "reg_lambda": 1.0, "reg_alpha": 0.0
    },
    "latin": {
        "max_depth": 4, "learning_rate": 0.08, "n_estimators": 300,
        "subsample": 0.9, "colsample_bytree": 0.9, "reg_lambda": 1.0, "reg_alpha": 0.0
    },
    "electronic": {
        "max_depth": 4, "learning_rate": 0.08, "n_estimators": 300,
        "subsample": 0.9, "colsample_bytree": 0.9, "reg_lambda": 1.0, "reg_alpha": 0.0
    },
}

# Grid for hybrid tuning (kept moderate for annual data sizes)
GRID = {
    "max_depth":         [3, 4, 5, 6],
    "learning_rate":     [0.03, 0.05, 0.08, 0.1],
    "n_estimators":      [200, 300, 400, 600],
    "subsample":         [0.7, 0.8, 0.9, 1.0],
    "colsample_bytree":  [0.7, 0.8, 0.9, 1.0],
    "reg_lambda":        [0.0, 0.5, 1.0, 2.0],
    "reg_alpha":         [0.0, 0.1, 0.5],
}

def _build_param_grid(grid):
    # Cartesian product of grid dict -> list of dicts
    import itertools
    keys = list(grid.keys())
    vals = [grid[k] for k in keys]
    combos = []
    for v in itertools.product(*vals):
        combos.append({k: v[i] for i, k in enumerate(keys)})
    return combos

GRID_COMBOS = _build_param_grid(GRID)

# ---------------
# Helper functions
# ---------------
def _windows(years, tr=3, te=2):
    """Return **all** contiguous windows (no cap)."""
    res, total = [], tr + te
    for i in range(0, len(years) - total + 1):
        a, b = years[i:i+tr], years[i+tr:i+total]
        if all(np.diff(a) == 1) and all(np.diff(b) == 1) and a[-1] + 1 == b[0]:
            res.append((a, b))
    return res

def _eval(y_true, y_pred):
    """Return RMSE, MAE, R2."""
    return (
        float(np.sqrt(mean_squared_error(y_true, y_pred))),
        float(mean_absolute_error(y_true, y_pred)),
        float(r2_score(y_true, y_pred)),
    )

In [ ]:
# -------------------------------------
# Load and prepare the aggregated data
# -------------------------------------
df = pd.read_csv('../Datasets/06_aggregated_dataset.csv')
df = df[df['Most_Prominent_Genre'].isin(FOCUS_GENRES)]
df = df[(df['Year'] >= YEAR_MIN) & (df['Year'] <= YEAR_MAX)]
df = df.rename(columns={'Most_Prominent_Genre': 'genre'})
df = df.sort_values(['genre', 'Year']).reset_index(drop=True)

print(f"Loaded {len(df)} rows for {len(df['genre'].unique())} genres")
df.head()

In [ ]:
# --------------------------------------------------------
# Initialize containers to collect results across genres
# --------------------------------------------------------
all_metrics, all_candidates, all_fi = [], [], []

In [ ]:
# ------------------------------------------------------
# Main loop: process each genre with walk-forward CV
# ------------------------------------------------------
for genre in FOCUS_GENRES:
    print(f"\n=== PROCESSING GENRE: {genre.upper()} ===")
    
    g = df[df['genre'] == genre].copy()
    years = sorted(g['Year'].unique())
    
    if len(years) < TRAIN_YEARS + TEST_YEARS:
        print(f"Skipping {genre}: insufficient data ({len(years)} years)")
        continue
    
    wins = _windows(years, TRAIN_YEARS, TEST_YEARS)
    print(f"Found {len(wins)} walk-forward windows")
    
    fold_metrics, fold_candidates, fold_fi = [], [], []
    
    # For each walk-forward window...
    for fold, (tr_years, te_years) in enumerate(wins, 1):
        # Split data
        tr = g[g['Year'].isin(tr_years)]
        te = g[g['Year'].isin(te_years)]
        
        X_tr, y_tr = tr[FEATURES], tr[TARGET]
        X_te, y_te = te[FEATURES], te[TARGET]
        
        # 1. Baseline: manual params
        mp = MANUAL_PARAMS[genre]
        m = XGBRegressor(random_state=RANDOM_STATE, **mp)
        m.fit(X_tr, y_tr)
        p = m.predict(X_te)
        rmse, mae, r2 = _eval(y_te, p)
        
        # Record manual param results
        fold_candidates.append({
            'genre': genre, 'fold': fold,
            'train_start': tr_years[0], 'train_end': tr_years[-1],
            'test_start': te_years[0], 'test_end': te_years[-1],
            'rmse': rmse, 'mae': mae, 'r2': r2,
            'source': 'manual', **mp
        })
        
        # 2. Grid search (modest, yearly data)
        best_rmse, best_p = rmse, p  # default to manual if grid fails
        
        for combo in GRID_COMBOS[:100]:  # Limit search space
            try:
                m = XGBRegressor(random_state=RANDOM_STATE, **combo)
                m.fit(X_tr, y_tr)
                p = m.predict(X_te)
                rmse, mae, r2 = _eval(y_te, p)
                
                fold_candidates.append({
                    'genre': genre, 'fold': fold,
                    'train_start': tr_years[0], 'train_end': tr_years[-1],
                    'test_start': te_years[0], 'test_end': te_years[-1],
                    'rmse': rmse, 'mae': mae, 'r2': r2,
                    'source': 'grid', **combo
                })
                
                if rmse < best_rmse:
                    best_rmse, best_p = rmse, p
            except Exception:
                continue
        
        # 3. Record final (best) fold results
        rmse, mae, r2 = _eval(y_te, best_p)
        fold_metrics.append({
            'genre': genre, 'fold': fold,
            'train_start': tr_years[0], 'train_end': tr_years[-1],
            'test_start': te_years[0], 'test_end': te_years[-1],
            'rmse': rmse, 'mae': mae, 'r2': r2
        })
        
        # Add winning params to that row
        winner_source = 'manual' if rmse == fold_candidates[-len(GRID_COMBOS[:100])-1]['rmse'] else 'grid'
        winner_row = [c for c in fold_candidates if c['genre']==genre and c['fold']==fold][-1]
        for k in ['max_depth','learning_rate','n_estimators','subsample','colsample_bytree','reg_lambda','reg_alpha']:
            fold_metrics[-1][k] = winner_row[k]
        
        # 4. Feature importance (from final model)
        final_combo = {k: winner_row[k] for k in ['max_depth','learning_rate','n_estimators',
                                                   'subsample','colsample_bytree','reg_lambda','reg_alpha']}
        m = XGBRegressor(random_state=RANDOM_STATE, **final_combo)
        m.fit(X_tr, y_tr)
        fi = m.feature_importances_
        
        for feat, imp in zip(FEATURES, fi):
            fold_fi.append({
                'genre': genre, 'fold': fold, 'feature': feat, 'importance': imp
            })
    
    # Collect genre results
    all_metrics.extend(fold_metrics)
    all_candidates.extend(fold_candidates)
    all_fi.extend(fold_fi)
    
    # Convert to DataFrames
    metrics_df = pd.DataFrame(fold_metrics)
    candidates_df = pd.DataFrame(fold_candidates)
    fi_df = pd.DataFrame(fold_fi)
    
    # Save per-genre outputs
    os.makedirs('../Models/outputs/xgb', exist_ok=True)
    metrics_df.to_csv(f'../Models/outputs/xgb/{genre}_xgb_walkforward_metrics.csv', index=False)
    candidates_df.to_csv(f'../Models/outputs/xgb/{genre}_xgb_wf_candidates.csv', index=False)
    fi_df.to_csv(f'../Models/outputs/xgb/{genre}_xgb_feature_importance_by_fold.csv', index=False)

In [ ]:
# ------------------------------
# Holdout: 2023–2024 vs final XGB
# ------------------------------
print("\n=== HOLDOUT EVALUATION (2023-2024) ===")

for genre in FOCUS_GENRES:
    g = df[df['genre'] == genre].copy()
    
    # Train on ≤2022
    tr = g[g['Year'] <= 2022]
    te = g[g['Year'].isin([2023, 2024])]
    
    if len(tr) == 0 or len(te) == 0:
        print(f"Skipping {genre} holdout: missing data")
        continue
    
    X_tr, y_tr = tr[FEATURES], tr[TARGET]
    X_te, y_te = te[FEATURES], te[TARGET]
    
    # Find final params (lowest avg RMSE across folds for this genre)
    genre_metrics = [m for m in all_metrics if m['genre'] == genre]
    if not genre_metrics:
        print(f"Skipping {genre} holdout: no metrics")
        continue
        
    # Group by param combo and average RMSE
    from collections import defaultdict
    rmse_by_params = defaultdict(list)
    
    for m in genre_metrics:
        key = tuple(m[k] for k in ['max_depth','learning_rate','n_estimators',
                                   'subsample','colsample_bytree','reg_lambda','reg_alpha'])
        rmse_by_params[key].append(m['rmse'])
    
    # Find combo with lowest average RMSE
    best_key = min(rmse_by_params.keys(), key=lambda k: np.mean(rmse_by_params[k]))
    best_avg_rmse = np.mean(rmse_by_params[best_key])
    
    # Reconstruct param dict
    param_names = ['max_depth','learning_rate','n_estimators',
                   'subsample','colsample_bytree','reg_lambda','reg_alpha']
    final_params = dict(zip(param_names, best_key))
    
    # Train final model
    m = XGBRegressor(random_state=RANDOM_STATE, **final_params)
    m.fit(X_tr, y_tr)
    p = m.predict(X_te)
    rmse, mae, r2 = _eval(y_te, p)
    
    # Save results
    holdout_results = pd.DataFrame({
        'Year': te['Year'].values,
        'Actual': y_te.values,
        'Predicted': p,
        'Residual': y_te.values - p
    })
    
    holdout_results.to_csv(f'../Models/outputs/xgb/{genre}_holdout_2023_2024_vs_xgb.csv', index=False)
    
    print(f"{genre.upper()}: RMSE={rmse:.2f}, MAE={mae:.2f}, R2={r2:.2f}")

In [ ]:
# ----------------------------
# Summary report and exports
# ----------------------------
print("\n=== SUMMARY REPORT ===")

# Aggregate metrics
metrics_df = pd.DataFrame(all_metrics)
candidates_df = pd.DataFrame(all_candidates)
fi_df = pd.DataFrame(all_fi)

# Per-genre averages
summary = metrics_df.groupby('genre').agg({
    'rmse': ['mean', 'std'],
    'mae': 'mean',
    'r2': 'mean',
    'fold': 'count'
}).round(3)

summary.columns = ['avg_rmse_walkforward', 'std_rmse_walkforward', 
                   'avg_mae_walkforward', 'avg_r2_walkforward', 'folds']
summary = summary.reset_index()

# Final params per genre (lowest avg RMSE combo)
from collections import defaultdict
for genre in summary['genre']:
    genre_metrics = metrics_df[metrics_df['genre'] == genre]
    if len(genre_metrics) == 0:
        continue
        
    # Group by param combo
    rmse_by_params = defaultdict(list)
    param_records = {}
    
    for _, row in genre_metrics.iterrows():
        key = tuple(row[k] for k in ['max_depth','learning_rate','n_estimators',
                                     'subsample','colsample_bytree','reg_lambda','reg_alpha'])
        rmse_by_params[key].append(row['rmse'])
        param_records[key] = {k: row[k] for k in ['max_depth','learning_rate','n_estimators',
                                                  'subsample','colsample_bytree','reg_lambda','reg_alpha']}
    
    # Find best
    if rmse_by_params:
        best_key = min(rmse_by_params.keys(), key=lambda k: np.mean(rmse_by_params[k]))
        best_params_str = str(param_records[best_key])
        meets_thresh = (
            np.mean(rmse_by_params[best_key]) <= THRESH_RMSE and
            metrics_df[metrics_df['genre'] == genre]['mae'].mean() <= THRESH_MAE and
            metrics_df[metrics_df['genre'] == genre]['r2'].mean() >= THRESH_R2
        )
        
        summary.loc[summary['genre'] == genre, 'final_params'] = best_params_str
        summary.loc[summary['genre'] == genre, 'meets_thresholds_walkforward'] = meets_thresh

# Save all outputs
summary.to_csv('../Models/outputs/xgb/xgb_summary_by_genre.csv', index=False)
candidates_df.to_csv('../Models/outputs/xgb/xgb_all_candidates.csv', index=False)
fi_df.to_csv('../Models/outputs/xgb/xgb_all_feature_importance.csv', index=False)

# Save grid used
with open('../Models/outputs/xgb/grid_xgb_used.txt', 'w') as f:
    f.write("Manual params per genre:\n")
    for genre, params in MANUAL_PARAMS.items():
        f.write(f"{genre}: {params}\n")
    f.write("\nGrid:\n")
    for combo in GRID_COMBOS[:100]:
        f.write(f"{combo}\n")

print(summary[['genre', 'folds', 'avg_rmse_walkforward', 'avg_mae_walkforward', 'avg_r2_walkforward']])
print("\nPipeline complete. Outputs saved to ../Models/outputs/xgb/")